# pMMR: Neighborhood analysis with just immune cells and based on comparisons of interest

Poportions will be calculated based on the subset of immune cells in each ROI.

In [102]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors
import time
import sys
import matplotlib.pyplot as plt
from sklearn.cluster import MiniBatchKMeans
import seaborn as sns
import math
import os
import re

from scipy.spatial import cKDTree
from scipy.spatial import distance_matrix
from scipy.stats import ttest_ind
from scipy.stats import ttest_rel
from itertools import combinations
import pandas as pd
import statsmodels.stats.multitest as smm

from scipy import stats


import matplotlib.cm as cm
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [103]:
import matplotlib.pyplot as plt
plt.rcdefaults()


I am working with these ROIs in here:

In [104]:
path = "/SSD/maa7095/spatial_techniques/manish_comet/sep_new_ROI/"

all_files = os.listdir(path)
pattern = re.compile(r'(.+)_sigma_binary_res')

# Extract matching sample names
sample_names = []
for file in all_files:
    match = pattern.match(file)
    if match:
        sample_names.append(match.group(1)) 

In [106]:
#create def to create a new table:
all_sample_merged_table= []
for sample_name in sample_names:
    annota_table = pd.read_table(f'/SSD/maa7095/spatial_techniques/manish_comet/sep_new_ROI/{sample_name}_sigma_binary_res/cell_annotation.txt')
    raw_table = pd.read_csv(f'/SSD/maa7095/spatial_techniques/manish_comet/sep_new_ROI/{sample_name}_sigma_binary_res/raw_metadata.csv')
    #print(annota_table['cell_id'].isin(raw_table['Object Id']).all())
    merged_table = pd.merge(
        raw_table,
        annota_table,
        left_on='Object Id',
        right_on='cell_id',
        how='inner'
    )
    merged_table['sample_region']=sample_name
    all_sample_merged_table.append(merged_table)

all_sample_merged_table = pd.concat(all_sample_merged_table, axis=0).reset_index(drop=True)

In [107]:
all_zscore_table= []
for sample_name in sample_names:
    norm_data = pd.read_csv(f'/SSD/maa7095/spatial_techniques/manish_comet/sep_new_ROI/{sample_name}_sigma_binary_res/z_score_normalization.csv')
    norm_data['sample_region']=sample_name
    all_zscore_table.append(norm_data)

all_zscore_table = pd.concat(all_zscore_table, axis=0).reset_index(drop=True)

In [108]:
all_sample_merged_table2 = all_sample_merged_table

In [109]:
all_sample_merged_table2.loc[all_sample_merged_table2['sample_region'] == 'NEST1-06_tumor_inner', 'sample_region'] = 'NEST1-06_tumor_bed'

In [110]:
normalized_df = all_sample_merged_table2.merge(all_zscore_table, on=['cell_id', 'sample_region'], how='left')


In [111]:
#rename cell types
probes_list =pd.read_csv("/SSD/maa7095/spatial_techniques/manish_comet/sep_new_ROI/probes_cell_type_table.csv",header=0)
normalized_df = pd.merge(normalized_df, probes_list, on="celll_type")

In [112]:
#select just the sample names, cell type of interest, X_mean and Y_mean and Cell Area
columns_to_Select = ['sample_region','new_cell_type_name','Cell Area (µm²)','xmean', 'ymean']
sub_df = normalized_df[columns_to_Select]

In [113]:
#sub_df

In [114]:
#they have decided to combine cell types.
sub_df["combined_names"]=sub_df["new_cell_type_name"]
sub_df.loc[sub_df['new_cell_type_name'] == 'CD4', 'combined_names'] = 'CD4_T_cell'
sub_df.loc[sub_df['new_cell_type_name'] == 'CD3E_CD4 (CD4_Tcell)', 'combined_names'] = 'CD4_T_cell'
sub_df.loc[sub_df['new_cell_type_name'] == 'CD4_LAG3_Tcell', 'combined_names'] = 'CD4_T_cell'
sub_df.loc[sub_df['new_cell_type_name'] == 'CD8', 'combined_names'] = 'CD8_T_cell'
sub_df.loc[sub_df['new_cell_type_name'] == 'CD3E_CD8 (CD8_Tcell)', 'combined_names'] = 'CD8_T_cell'
sub_df.loc[sub_df['new_cell_type_name'] == 'CD8_LAG3_Tcell', 'combined_names'] = 'CD8_T_cell'
sub_df.loc[sub_df['new_cell_type_name'] == 'CD3E_CD4_CD8 (Tcell)', 'combined_names'] = 'T_cell'
sub_df.loc[sub_df['new_cell_type_name'] == 'LAG3_Tcell', 'combined_names'] = 'T_cell'
sub_df.loc[sub_df['new_cell_type_name'] == 'FOXP3', 'combined_names'] = 'T_reg'
sub_df.loc[sub_df['new_cell_type_name'] == 'CD3E_FOXP3 (T_regs)', 'combined_names'] = 'T_reg'
sub_df.loc[sub_df['new_cell_type_name'] == 'CD3E_CD8_FOXP3 (CD8_Tregs)', 'combined_names'] = 'CD8_Treg'
sub_df.loc[sub_df['new_cell_type_name'] == 'CD3E_CD4_FOXP3 (CD4_Tregs)', 'combined_names'] = 'CD4_Treg'
sub_df.loc[sub_df['new_cell_type_name'] == 'Macrophages', 'combined_names'] = 'M1_Macrophages'
sub_df.loc[sub_df['new_cell_type_name'] == 'Th2_Macrophage', 'combined_names'] = 'M2_Macrophages'

/tmp/ipykernel_3977106/3522644170.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sub_df["combined_names"]=sub_df["new_cell_type_name"]


In [115]:
#sub_df["combined_names"].value_counts()

In [116]:
celltype_colors = {
    "NK": "#1f77b4",
    "CD8_T_cell": "#2ca02c",
    "Endothelial": "#d62728",
    "Unknown": "#843c39",
    "B_cell": "#8c564b",
    "T_reg": "#e377c2",
    "Lymphatic_endothelium": "#7f7f7f",
    "Smooth_muscle": "#aec7e8",
    "CD8_Treg": "#ffbb78",
    "CD4_T_cell": "#98df8a",
    "M1_Macrophages": "#c5b0d5",
    "T_cell": "#c49c94",
    "Epithelial": "#f7b6d2",
    "CD4_Treg": "#393b79",
    "Stromal": "#637939",
    "M2_Macrophages": "#ff9896",
    "Plasma": "#9467bd"
}

## Subset for just immune cells.

In [117]:
immune_cells = ['B_cell', 'CD4_T_cell', 'CD4_Treg','CD8_T_cell', 'CD8_Treg','Plasma',"T_reg", 'M1_Macrophages', 'M2_Macrophages']

In [118]:
sub_df_immune = sub_df[sub_df['combined_names'].isin(immune_cells)]

## Subset for only just pMMR

In [119]:
samples_to_remove = ['NEST1-05_tumor','NEST1-05_tumor_bed','NEST1-06_tumor_bed']

In [120]:
sub_df_immune = sub_df_immune[~sub_df_immune['sample_region'].isin(samples_to_remove)]


## Number of immune cells per sample

In [ ]:
pd.crosstab(sub_df_immune['sample_region'], sub_df_immune['combined_names'])


### Total number of immune cells per genotype

In [ ]:
sub_df_immune['sample_region'].value_counts()

In [123]:
regions = ["tumor_inner", "tumor_bed", "tumor"]

# Create regex pattern
pattern = re.compile(rf"({'|'.join(regions)})$")

# Split into sample_id and region
data = []
for s in sub_df_immune['sample_region']:
    match = pattern.search(s)
    if match:
        region = match.group(1)
        sample_id = s.replace("_" + region, "")
    else:
        region = "other"
        sample_id = s
    data.append((sample_id, region))

df = pd.DataFrame(data, columns=["Sample_name", "ROI"])

In [124]:
sub_df_immune['Sample_name']=df['Sample_name'].values
sub_df_immune['ROI']=df['ROI'].values

In [ ]:
sample_map = pd.read_csv("marker_files/sample_conditions.csv")  
sample_map

In [126]:
sub_df_immune = pd.merge(
    sub_df_immune, sample_map,
    left_on=["Sample_name", "ROI"],
    right_on=["Sample_name", "ROI"],
    how="inner"
)


# Spatial nieghbors

## Spatial nieghbors by Conditions

Here we are computing for each cell type which is the proportion of other cell types wihin 50 microns. Based on differnet conditions.

In [129]:
np.random.seed(42)

def neighbor_enrichment(df, sample_col="sample_region", cell_type_col="new_cell_type_name", radius=20.0, cond=None):
    """
    Computes the counts and proportions of neighbor cell types within a given radius
    for each cell type in the dataset, aggregated over all samples.

    Returns:
        neighbor_summary_df: DataFrame with cell types as rows, neighbor types as columns,
                             values are proportion of neighbors.
    """
    neighbor_summary = []

    for cell_type_interest in df[cell_type_col].unique():
        all_neighbors = []

        for sample in df[sample_col].unique():
            df_sample = df[df[sample_col] == sample]
            coords_sample = df_sample[['xmean', 'ymean']].values

            is_interest = df_sample[cell_type_col] == cell_type_interest
            interest_coords = coords_sample[is_interest.values]

            if len(interest_coords) == 0:
                continue

            other_coords = coords_sample[~is_interest.values]
            other_cell_types = df_sample.loc[~is_interest, cell_type_col].values

            if len(other_coords) == 0:
                continue
            #cKDTree → efficient spatial index for nearest-neighbor search.
            #query_ball_point → returns indices of all other cells within radius for each cell of interest.
            tree = cKDTree(other_coords)
            neighbor_idx_list = tree.query_ball_point(interest_coords, r=radius)

            # Flatten neighbor types
            for neighbors in neighbor_idx_list:
                if len(neighbors) > 0:
                    all_neighbors.extend(other_cell_types[neighbors])

        # Count and normalize
        neighbor_counts = pd.Series(all_neighbors).value_counts()
        neighbor_proportions = neighbor_counts / neighbor_counts.sum() if len(neighbor_counts) > 0 else pd.Series()

        # Add cell type info
        for neighbor_type, prop in neighbor_proportions.items():
            neighbor_summary.append({
                "cell_type": cell_type_interest,
                "neighbor_cell_type": neighbor_type,
                "proportion": prop
            })

    neighbor_summary_df = pd.DataFrame(neighbor_summary)

    # Optional: pivot for heatmap
    pivot_df = neighbor_summary_df.pivot(index="cell_type", columns="neighbor_cell_type", values="proportion").fillna(0)

    # Plot heatmap
    plt.figure(figsize=(10, 6))
    sns.heatmap(pivot_df, cmap="viridis", annot=False, fmt=".2f")
    plt.xlabel("Neighbor cell type")
    plt.ylabel("Cell type of interest")
    plt.title(f"Neighbor enrichment within {radius} microns for {cond}")
    plt.tight_layout()
    plt.show()

    return neighbor_summary_df, pivot_df


In [ ]:
conditions = sub_df_immune['Condition'].unique()

neighbor_dfs = {}  # Store results per condition
pivot_dfs = {}

for cond in conditions:
    df_cond = sub_df_immune[sub_df_immune['Condition'] == cond]
    print(f"Samples to compute the neighbor analysis: {df_cond['sample_region'].unique()}")
    neighbor_summary_df, pivot_df = neighbor_enrichment(
        df_cond,
        sample_col="sample_region",
        cell_type_col="combined_names",
        radius=50,
        cond=cond
    )
    
    neighbor_dfs[cond] = neighbor_summary_df
    pivot_dfs[cond] = pivot_df

## Spatial neighbors by ROI

In [131]:
sub_df_immune.loc[sub_df_immune['sample_region'] == 'NEST1-04_tumor', 'ROI'] = 'tumor_inner'


In [ ]:
conditions = sub_df_immune['ROI'].unique()

neighbor_dfs = {}  # Store results per condition
pivot_dfs = {}

for cond in conditions:
    df_cond = sub_df_immune[sub_df_immune['ROI'] == cond]
    print(f"Samples to compute the neighbor analysis: {df_cond['sample_region'].unique()}")
    neighbor_summary_df, pivot_df = neighbor_enrichment(
        df_cond,
        sample_col="sample_region",
        cell_type_col="combined_names",
        radius=50,
        cond=cond
    )
    
    neighbor_dfs[cond] = neighbor_summary_df
    pivot_dfs[cond] = pivot_df

In [133]:
sub_df_immune.loc[sub_df_immune['sample_region'] == 'NEST1-04_tumor', 'ROI'] = 'tumor'


In [134]:
conditions_interest = ["Responder","Non-responder"]
ROI_interest = ["tumor_bed","tumor_inner"]

In [135]:
sub_interest1 = sub_df_immune[sub_df_immune['Condition'].isin(conditions_interest)]
sub_interest1 = sub_interest1[sub_interest1['ROI'].isin(ROI_interest)]


## MSI stable: all Responders and Non-responders

In [136]:
samples_to_subset = ['NEST1-06_tumor_bed','NEST1-01_tumor_bed',
       'NEST2-03_tumor_bed', 'NEST1-11_tumor_bed',
       'NEST2-01_tumor_bed','NEST2-06_tumor_bed']
MSI_samples = sub_df_immune[sub_df_immune['sample_region'].isin(samples_to_subset)]
MSI_samples['sample_region'].unique()

array(['23257_A7_T3_tumor_bed', 'S23-47115_A6_tumor_bed',
       'S23-18584_A5_tumor_bed', 'S24-21915_A5_tumor_bed',
       'S23-21917_A6_tumor_bed'], dtype=object)

In [ ]:
conditions = MSI_samples['MSI'].unique()

neighbor_dfs = {}  # Store results per condition
pivot_dfs = {}

for cond in conditions:
    df_cond = MSI_samples[MSI_samples['MSI'] == cond]
    print(f"Samples to compute the neighbor analysis: {df_cond['sample_region'].unique()}")
    neighbor_summary_df, pivot_df = neighbor_enrichment(
        df_cond,
        sample_col="sample_region",
        cell_type_col="combined_names",
        radius=50,
        cond=cond
    )
    
    neighbor_dfs[cond] = neighbor_summary_df
    pivot_dfs[cond] = pivot_df

## In Responders: MSI stable

In [138]:
MSI_RESPONDERS = MSI_samples[MSI_samples['Condition']=='Responder']
conditions = MSI_RESPONDERS['MSI'].unique()
conditions


array(['Stable'], dtype=object)

In [ ]:
neighbor_dfs = {}  # Store results per condition
pivot_dfs = {}

for cond in conditions:
    df_cond = MSI_RESPONDERS[MSI_RESPONDERS['MSI'] == cond]
    print(f"Samples to compute the neighbor analysis: {df_cond['sample_region'].unique()}")
    neighbor_summary_df, pivot_df = neighbor_enrichment(
        df_cond,
        sample_col="sample_region",
        cell_type_col="combined_names",
        radius=50,
        cond=cond
    )
    
    neighbor_dfs[cond] = neighbor_summary_df
    pivot_dfs[cond] = pivot_df

## Mutation status
In Non-responders WT, KRAS and BRAF

In [140]:
samples_to_subset = ['NEST2-10_tumor_inner','NEST2-04_tumor_inner','NEST1-08_tumor_inner','NEST1-04_tumor']
mutation_Samples = sub_df_immune[sub_df_immune['sample_region'].isin(samples_to_subset)]
mutation_Samples['mutation_state']='WT'
mutation_Samples.loc[mutation_Samples['sample_region'] == 'NEST2-04_tumor_inner', 'mutation_state'] = 'BRAF_V600R'
mutation_Samples.loc[mutation_Samples['sample_region'] == 'NEST2-10_tumor_inner', 'mutation_state'] = 'KRAS_G12D'


/tmp/ipykernel_3977106/268937793.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  mutation_Samples['mutation_state']='WT'


In [ ]:
conditions = mutation_Samples['mutation_state'].unique()
neighbor_dfs = {}  # Store results per condition
pivot_dfs = {}

for cond in conditions:
    df_cond = mutation_Samples[mutation_Samples['mutation_state'] == cond]
    print(f"Samples to compute the neighbor analysis: {df_cond['sample_region'].unique()}")
    neighbor_summary_df, pivot_df = neighbor_enrichment(
        df_cond,
        sample_col="sample_region",
        cell_type_col="combined_names",
        radius=50,
        cond=cond
    )
    
    neighbor_dfs[cond] = neighbor_summary_df
    pivot_dfs[cond] = pivot_df

## Calculate distances between specific cell types for Respobders vs Non-responders

In [142]:
#sub_df_immune['combined_names'].unique()

In [143]:
np.random.seed(42)

def nn_dists(coords, source_idx, target_idx, k=1):
    """Compute nearest neighbor distances from source to target coordinates."""
    if len(source_idx) == 0 or len(target_idx) == 0:
        return np.array([])
    tree = cKDTree(coords[target_idx])
    dists, _ = tree.query(coords[source_idx], k=k)
    return dists if k > 1 else dists.ravel()


# --- Compute per-sample distances ---
results = []
radius = 50.0  # microns or pixels; adjust to your spatial scale



In [144]:
results = []

for sample in sub_df_immune['sample_region'].unique():
    sub = sub_df_immune[sub_df_immune['sample_region'] == sample].reset_index(drop=True)
    coords = sub[['xmean', 'ymean']].to_numpy()

    # all unique cell types present in this sample
    cell_types = sub['combined_names'].unique()

    # precompute indices for each cell type
    idx_map = {
        ct: np.where(sub['combined_names'] == ct)[0]
        for ct in cell_types
    }

    # iterate over every ordered pair of distinct cell types
    for src in cell_types:
        for tgt in cell_types:
            if src == tgt:
                continue  # skip self-distances

            dists = nn_dists(coords, idx_map[src], idx_map[tgt], k=1)

            def summarize(metric, dists):
                if len(dists) == 0:
                    return {
                        'sample': sample,
                        'metric': metric,
                        'n_source': 0,
                        'mean': np.nan,
                        'median': np.nan,
                        'prop_within_R': np.nan,
                    }
                return {
                    'sample': sample,
                    'metric': metric,
                    'n_source': len(dists),
                    'mean': float(np.nanmean(dists)),
                    'median': float(np.nanmedian(dists)),
                    'prop_within_R': float(np.mean(dists <= radius)),
                }

            metric_name = f"{src}_to_{tgt}"
            results.append(summarize(metric_name, dists))

res_df = pd.DataFrame(results)


In [145]:
sample_map['sample'] = sample_map['Sample_name'].astype(str) + '_' + sample_map['ROI'].astype(str)

In [146]:
res_df = pd.merge(
    res_df, sample_map,
    left_on=["sample"],
    right_on=["sample"],
    how="inner"
)
 

In [147]:
res_df.loc[res_df['sample'] == 'NEST1-04_tumor', 'ROI'] = 'tumor_inner'


In [148]:
keep_roi = ['tumor_bed','tumor_inner']
sub_res_Df = res_df[res_df['ROI'].isin(keep_roi)]

In [ ]:
sub_res_Df['sample'].unique()

### Table of the samples used in this analysis

In [ ]:
pd.crosstab(sub_res_Df['sample'], sub_res_Df['Condition'])


In [ ]:
pd.crosstab(sub_res_Df['sample'], sub_res_Df['ROI'])


In [152]:
sub_metrics = ['CD8_T_cell_to_T_reg','CD8_T_cell_to_Plasma','B_cell_to_CD8_T_cell','B_cell_to_CD4_T_cell','B_cell_to_Plasma']

In [153]:
mask = sub_res_Df['metric'].str.startswith('T_reg')
t_Reg_metrics = sub_res_Df['metric'][mask].unique()
sub_metrics = t_Reg_metrics.tolist() + sub_metrics

In [154]:
sub_sub_res_Df = sub_res_Df[sub_res_Df['metric'].isin(sub_metrics)]

1. In Fig 4B (for non-responders) we suggest that CD8 cells have a local microenvironment enriched with FOXP3+ and CD8 Tregs, whereas in 4C (for responders), it looks like CD8 cells are surrounded more by plasma cells and CD4 cells.  Can we calculate the distances of these cell types to CD8+ cells?  Similar for the association between B cells and CD8 cells/ plasma cells and CD4 cells in responders v non responders?

In [ ]:
order_conditions = ['Non-responder','Responder']
for metric in sub_sub_res_Df['metric'].unique():
    sub = sub_sub_res_Df[sub_sub_res_Df['metric'] == metric]
    print(sub.groupby('Condition')['mean'].mean())
    groups = [c for c in order_conditions if c in sub['Condition'].unique()]
    flierprops = dict(marker='o', markersize=0, linestyle='none')  # size 0 removes them
    plt.figure(figsize=(8,6))
    ax = sns.boxplot(
        data=sub,
        x='Condition',
        y='mean',
        palette='Set2',
        order=groups,
        flierprops=flierprops
    )
    sns.stripplot(
        data=sub,
        x='Condition',
        y='mean',
        color='black',
        size=10,
        jitter=True,
        ax=ax,
        order=groups,
    )
    
    # Compute pairwise comparisons and annotate
    y_max = sub['mean'].max()
    h = y_max * 0.05  # height increment for annotations
    start = y_max + h
    step = h * 1.5
    
    for i, (g1, g2) in enumerate(combinations(groups, 2)):
        a = sub.loc[sub['Condition'] == g1, 'mean'].dropna()
        b = sub.loc[sub['Condition'] == g2, 'mean'].dropna()
        if len(a) > 0 and len(b) > 0:
            stat, p = stats.ttest_ind(a, b, equal_var=False)
            x1, x2 = list(groups).index(g1), list(groups).index(g2)
            y = start + step * i
            ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c='k')
            ax.text((x1+x2)*.5, y+h, f"p={p:.3g}", ha='center', va='bottom', color='k')
    
    plt.title(f"{metric} per Condition")
    plt.ylabel("Mean distance (µm)")
    plt.xlabel("Condition")
    plt.tight_layout()
    plt.show()



# Immune cell type proportion differences

## First comparison: Responders vs Non-responders
Just internal tumor and no Peritumoral area

In [156]:
interest_sample_sub = sub_df_immune[sub_df_immune['sample_region'] == "NEST1-04_tumor"]
sub_interest1 = pd.concat([sub_interest1, interest_sample_sub], axis=0)

In [157]:
sub_interest1['Condition'].unique()

array(['Responder', 'Non-responder'], dtype=object)

In [158]:
def t_test_plot(dataframe_interest, columns_interest, sample_order=None, t_test_type=ttest_ind):
    prop_table = pd.crosstab(
        [dataframe_interest[columns_interest], dataframe_interest['sample_region']], 
        dataframe_interest['combined_names'], 
        normalize='index'
    )
    prop_table.index.names = [columns_interest, 'sample_region']
    if sample_order is not None:
        # Reorder rows by desired condition order
        cond_order = pd.Categorical(
            prop_table.index.get_level_values(columns_interest),
            categories=sample_order,
            ordered=True
        )
        prop_table = prop_table.iloc[np.argsort(cond_order)]
         
    colors = [celltype_colors.get(ct, "#808080")  # fallback = grey if not in dict
              for ct in prop_table.columns]

    ax = prop_table.plot(kind='barh', stacked=True, figsize=(10, 6), color=colors)
    
    # add separator lines between conditions
    # get index levels
    conditions = prop_table.index.get_level_values(0)
    for i in range(1, len(conditions)):
        if conditions[i] != conditions[i-1]:
            ax.axhline(i-0.5, color="black", linewidth=1)  # separator
    
    # legend outside
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.grid(False)
    
    plt.title('Proportion of Cell Types per Sample')
    plt.xlabel('Proportion of Cells')
    plt.ylabel('Sample (grouped by Condition)')
    plt.tight_layout()
    plt.show()



    prop_table[columns_interest] = prop_table.index.get_level_values(columns_interest)
    results = []

    conditions = prop_table[columns_interest].unique()
    
    for cell_type in prop_table.columns:
        if cell_type != columns_interest:  # skip metadata
            # Loop over all pairwise comparisons
            for cond1, cond2 in combinations(conditions, 2):
                group1 = prop_table.loc[prop_table[columns_interest] == cond1, cell_type]
                group2 = prop_table.loc[prop_table[columns_interest] == cond2, cell_type]
    
                mean1 = group1.mean()
                mean2 = group2.mean()

                if t_test_type == ttest_rel:
                    t_stat, p_val = t_test_type(group1, group2)
                else:
                    t_stat, p_val = t_test_type(group1, group2, equal_var=False)
    
                #t_stat, p_val = t_test_type(group1, group2, equal_var=False)
    
                direction = f"up_in_{cond1}" if mean1 > mean2 else f"up_in_{cond2}"
    
                results.append({
                    "cell_type": cell_type,
                    "comparison": f"{cond1}_vs_{cond2}",
                    f"{cond1}_mean": mean1,
                    f"{cond2}_mean": mean2,
                    "t_stat": t_stat,
                    "p_val": p_val,
                    "direction": direction
                })
    
    # Convert to DataFrame
    results_df = pd.DataFrame(results)
    
    # Correct for multiple testing (FDR)
    results_df["p_adj"] = smm.multipletests(results_df["p_val"], method="fdr_bh")[1]
    
    return ax, results_df

In [ ]:
ax, results_df = t_test_plot(sub_interest1, "Condition",  sample_order=["Responder", "Non-responder"])

In [160]:
results_df[results_df['p_val']<0.05]

,cell_type,comparison,Responder_mean,Non-responder_mean,t_stat,p_val,direction,p_adj


### All the p values for ecah comparison

In [161]:
results_df

,cell_type,comparison,Responder_mean,Non-responder_mean,t_stat,p_val,direction,p_adj
0,B_cell,Responder_vs_Non-responder,0.383097,0.058069,2.690793,0.050708,up_in_Responder,0.418421
1,CD4_T_cell,Responder_vs_Non-responder,0.022718,0.217302,-2.151579,0.119803,up_in_Non-responder,0.418421
2,CD4_Treg,Responder_vs_Non-responder,0.000113,0.033589,-1.907638,0.152477,up_in_Non-responder,0.418421
3,CD8_T_cell,Responder_vs_Non-responder,0.273804,0.147983,0.947256,0.385752,up_in_Responder,0.495967
4,CD8_Treg,Responder_vs_Non-responder,0.000906,0.000831,0.079162,0.939708,up_in_Responder,0.939708
5,M1_Macrophages,Responder_vs_Non-responder,0.019745,0.053746,-1.220243,0.278948,up_in_Non-responder,0.418421
6,M2_Macrophages,Responder_vs_Non-responder,0.116584,0.173203,-0.275355,0.792994,up_in_Non-responder,0.892118
7,Plasma,Responder_vs_Non-responder,0.028901,0.003253,1.361435,0.243625,up_in_Responder,0.418421
8,T_reg,Responder_vs_Non-responder,0.154132,0.312024,-1.379229,0.228515,up_in_Non-responder,0.418421


### Ratio of CD8_T_cell and Tregs

In [ ]:
flierprops = dict(marker='o', markersize=0, linestyle='none')


values_per_Sample = pd.crosstab(
    [sub_interest1['Condition'], sub_interest1['sample_region']],
    sub_interest1['combined_names'],
)

numerator = 'CD8_T_cell'
denominator = 'T_reg'


ratios = (
    values_per_Sample[numerator] /
    values_per_Sample[denominator].replace(0, np.nan)
).rename('ratio')

ratios_df = ratios.reset_index()

print(ratios_df.groupby('Condition')['ratio'].mean())


fig, ax = plt.subplots(figsize=(6, 5))

sns.boxplot(
    data=ratios_df,
    x='Condition',
    y='ratio',
    palette='Set2',
    flierprops=flierprops,
    log_scale=True,
    ax=ax
)

sns.stripplot(
    data=ratios_df,
    x='Condition',
    y='ratio',
    color='black',
    alpha=0.6,
    jitter=True,
    size=10,
    ax=ax
)

ax.set_title(f'{numerator} / {denominator} ratio per Condition')
ax.set_ylabel('Log Ratio')
ax.set_xlabel('')


test_df = ratios_df.dropna(subset=['ratio'])
groups = test_df['Condition'].unique()

y_max = test_df['ratio'].max()
h = y_max * 0.05
start = y_max + h
step = h * 1.5

for i, (g1, g2) in enumerate(combinations(groups, 2)):
    a = test_df.loc[test_df['Condition'] == g1, 'ratio']
    b = test_df.loc[test_df['Condition'] == g2, 'ratio']

    if len(a) > 0 and len(b) > 0:
        stat, p = ttest_ind(a, b, equal_var=False)

        x1, x2 = list(groups).index(g1), list(groups).index(g2)
        y = start + step * i

        ax.plot(
            [x1, x1, x2, x2],
            [y, y + h, y + h, y],
            lw=1.5,
            c='k'
        )

        ax.text(
            (x1 + x2) * 0.5,
            y + h,
            f"p={p:.3g}",
            ha='center',
            va='bottom',
            color='k'
        )

plt.tight_layout()
plt.show()


In [ ]:
ratios_df

## Third comparison: Baseline vs Resections: paired-test

In [164]:
sub_df_immune['Internal'].unique()

array(['Resection', 'Baseline', nan], dtype=object)

In [ ]:
pd.crosstab(sub_df_immune['sample_region'], sub_df_immune['Internal'])

In [166]:
sub_interest1 = sub_df_immune[sub_df_immune['Internal'].notna()]

In [ ]:
sub_interest1['Sample_name'].unique()

In [168]:
sub_interest1 = sub_interest1[sub_interest1['Sample_name']!="NEST2-08a"]

In [ ]:
ax, results_df = t_test_plot(sub_interest1, "Internal", sample_order=["Resection", "Baseline"],t_test_type= ttest_rel)

In [170]:
results_df[results_df['p_val']<0.05]

,cell_type,comparison,Resection_mean,Baseline_mean,t_stat,p_val,direction,p_adj


In [171]:
results_df

,cell_type,comparison,Resection_mean,Baseline_mean,t_stat,p_val,direction,p_adj
0,B_cell,Resection_vs_Baseline,0.138514,0.136129,0.043637,0.967935,up_in_Resection,0.967935
1,CD4_T_cell,Resection_vs_Baseline,0.149448,0.197433,-0.295133,0.787139,up_in_Baseline,0.885531
2,CD4_Treg,Resection_vs_Baseline,0.009799,0.049939,-2.102379,0.126279,up_in_Baseline,0.504837
3,CD8_T_cell,Resection_vs_Baseline,0.236595,0.060421,1.811187,0.167791,up_in_Resection,0.504837
4,CD8_Treg,Resection_vs_Baseline,0.001204,0.001960,-0.686693,0.541594,up_in_Baseline,0.696335
5,M1_Macrophages,Resection_vs_Baseline,0.032896,0.071915,-1.526269,0.224372,up_in_Baseline,0.504837
6,M2_Macrophages,Resection_vs_Baseline,0.146328,0.000457,1.007360,0.387970,up_in_Resection,0.622203
7,Plasma,Resection_vs_Baseline,0.028165,0.085190,-0.944023,0.414802,up_in_Baseline,0.622203
8,T_reg,Resection_vs_Baseline,0.257051,0.396557,-1.773872,0.174187,up_in_Baseline,0.504837


### Ratio of CD8_T_cell and Tregs paired-test

In [ ]:
flierprops = dict(marker='o', markersize=0, linestyle='none')


values_per_Sample = pd.crosstab(
    [sub_interest1['Internal'], sub_interest1['sample_region']],
    sub_interest1['combined_names'],
)

numerator = 'CD8_T_cell'
denominator = 'T_reg'


ratios = (
    values_per_Sample[numerator] /
    values_per_Sample[denominator].replace(0, np.nan)
).rename('ratio')

ratios_df = ratios.reset_index()

print(ratios_df.groupby('Internal')['ratio'].mean())


fig, ax = plt.subplots(figsize=(6, 5))

sns.boxplot(
    data=ratios_df,
    x='Internal',
    y='ratio',
    palette='Set2',
    flierprops=flierprops,
    log_scale=True,
    ax=ax
)

sns.stripplot(
    data=ratios_df,
    x='Internal',
    y='ratio',
    color='black',
    alpha=0.6,
    jitter=True,
    size=10,
    ax=ax
)

ax.set_title(f'{numerator} / {denominator} ratio per Condition')
ax.set_ylabel('Log Ratio')
ax.set_xlabel('')


test_df = ratios_df.dropna(subset=['ratio'])
groups = test_df['Internal'].unique()

y_max = test_df['ratio'].max()
h = y_max * 0.05
start = y_max + h
step = h * 1.5

for i, (g1, g2) in enumerate(combinations(groups, 2)):
    a = test_df.loc[test_df['Internal'] == g1, 'ratio']
    b = test_df.loc[test_df['Internal'] == g2, 'ratio']

    if len(a) > 0 and len(b) > 0:
        stat, p = ttest_ind(a, b)

        x1, x2 = list(groups).index(g1), list(groups).index(g2)
        y = start + step * i

        ax.plot(
            [x1, x1, x2, x2],
            [y, y + h, y + h, y],
            lw=1.5,
            c='k'
        )

        ax.text(
            (x1 + x2) * 0.5,
            y + h,
            f"p={p:.3g}",
            ha='center',
            va='bottom',
            color='k'
        )

plt.tight_layout()
plt.show()

In [ ]:
ratios_df

## 4 comparison: Baseline vs Resections for Responder: paired-test


In [ ]:
test_to_subset = ['Responder','Baseline']
subset_table = sub_interest1[sub_interest1['Condition'].isin(test_to_subset)]
non_Responder = ['NEST1-04_tumor','NEST2-04_tumor']
subset_table = subset_table[~subset_table['sample_region'].isin(non_Responder)]
subset_table['sample_region'].unique()

In [175]:
subset_table = subset_table[subset_table['Sample_name']!="NEST2-08a"]

In [ ]:
ax, results_df = t_test_plot(subset_table, "Internal", sample_order=["Resection", "Baseline"] ,t_test_type= ttest_rel)

In [177]:
results_df[results_df['p_val']<0.05]

,cell_type,comparison,Resection_mean,Baseline_mean,t_stat,p_val,direction,p_adj


### Ratio of CD8_T_cell and Tregs paired-test

In [ ]:
flierprops = dict(marker='o', markersize=0, linestyle='none')


values_per_Sample = pd.crosstab(
    [subset_table['Internal'], subset_table['sample_region']],
    subset_table['combined_names'],
)

numerator = 'CD8_T_cell'
denominator = 'T_reg'


ratios = (
    values_per_Sample[numerator] /
    values_per_Sample[denominator].replace(0, np.nan)
).rename('ratio')

ratios_df = ratios.reset_index()

print(ratios_df.groupby('Internal')['ratio'].mean())


fig, ax = plt.subplots(figsize=(6, 5))

sns.boxplot(
    data=ratios_df,
    x='Internal',
    y='ratio',
    palette='Set2',
    flierprops=flierprops,
    log_scale=True,
    ax=ax
)

sns.stripplot(
    data=ratios_df,
    x='Internal',
    y='ratio',
    color='black',
    alpha=0.6,
    jitter=True,
    size=10,
    ax=ax
)

ax.set_title(f'{numerator} / {denominator} ratio per Condition')
ax.set_ylabel('Log Ratio')
ax.set_xlabel('')


test_df = ratios_df.dropna(subset=['ratio'])
groups = test_df['Internal'].unique()

y_max = test_df['ratio'].max()
h = y_max * 0.05
start = y_max + h
step = h * 1.5

for i, (g1, g2) in enumerate(combinations(groups, 2)):
    a = test_df.loc[test_df['Internal'] == g1, 'ratio']
    b = test_df.loc[test_df['Internal'] == g2, 'ratio']

    if len(a) > 0 and len(b) > 0:
        stat, p = ttest_ind(a, b)

        x1, x2 = list(groups).index(g1), list(groups).index(g2)
        y = start + step * i

        ax.plot(
            [x1, x1, x2, x2],
            [y, y + h, y + h, y],
            lw=1.5,
            c='k'
        )

        ax.text(
            (x1 + x2) * 0.5,
            y + h,
            f"p={p:.3g}",
            ha='center',
            va='bottom',
            color='k'
        )

plt.tight_layout()
plt.show()

In [ ]:
ratios_df

## 5 comparison: Baseline vs Resections for Non-responder: paired-test


In [ ]:
test_to_subset = ['Non-responder','Baseline']
subset_table = sub_interest1[sub_interest1['Condition'].isin(test_to_subset)]
Responder = ['NEST1-05_tumor']
subset_table = subset_table[~subset_table['sample_region'].isin(Responder)]
subset_table['sample_region'].unique()

In [ ]:
ax, results_df = t_test_plot(subset_table, "Internal", sample_order=["Resection", "Baseline"] ,t_test_type= ttest_rel)

In [182]:
results_df[results_df['p_val']<0.05]

,cell_type,comparison,Resection_mean,Baseline_mean,t_stat,p_val,direction,p_adj


### Ratio of CD8_T_cell and Tregs paired-test

In [ ]:
flierprops = dict(marker='o', markersize=0, linestyle='none')


values_per_Sample = pd.crosstab(
    [subset_table['Internal'], subset_table['sample_region']],
    subset_table['combined_names'],
)

numerator = 'CD8_T_cell'
denominator = 'T_reg'


ratios = (
    values_per_Sample[numerator] /
    values_per_Sample[denominator].replace(0, np.nan)
).rename('ratio')

ratios_df = ratios.reset_index()

print(ratios_df.groupby('Internal')['ratio'].mean())


fig, ax = plt.subplots(figsize=(6, 5))

sns.stripplot(
    data=ratios_df,
    x='Internal',
    y='ratio',
    color='black',
    alpha=1,
    jitter=True,
    palette='Set2', 
    size=10
)


ax.set_title(f'{numerator} / {denominator} ratio per Condition')
ax.set_ylabel('Log Ratio')
ax.set_xlabel('')


test_df = ratios_df.dropna(subset=['ratio'])
groups = test_df['Internal'].unique()

y_max = test_df['ratio'].max()
h = y_max * 0.05
start = y_max + h
step = h * 1.5

for i, (g1, g2) in enumerate(combinations(groups, 2)):
    a = test_df.loc[test_df['Internal'] == g1, 'ratio']
    b = test_df.loc[test_df['Internal'] == g2, 'ratio']

    if len(a) > 0 and len(b) > 0:
        stat, p = ttest_ind(a, b)

        x1, x2 = list(groups).index(g1), list(groups).index(g2)
        y = start + step * i

        ax.plot(
            [x1, x1, x2, x2],
            [y, y + h, y + h, y],
            lw=1.5,
            c='k'
        )

        ax.text(
            (x1 + x2) * 0.5,
            y + h,
            f"p={p:.3g}",
            ha='center',
            va='bottom',
            color='k'
        )

plt.tight_layout()
plt.show()

In [ ]:
ratios_df

## 6 comparison: within Baseline: Responders vs Non-responders

In [185]:
sub_interest1 = sub_df_immune[sub_df_immune['Internal']=="Baseline"]

In [186]:
sub_interest1['Baseline_comp']='Responder'
sub_interest1.loc[sub_interest1['sample_region'] == 'NEST2-04_tumor', 'Baseline_comp'] = 'Non-responder'
sub_interest1.loc[sub_interest1['sample_region'] == 'NEST1-04_tumor', 'Baseline_comp'] = 'Non-responder'

/tmp/ipykernel_3977106/2083488142.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sub_interest1['Baseline_comp']='Responder'


In [ ]:
ax, results_df = t_test_plot(sub_interest1, "Baseline_comp")

### Ratio of CD8_T_cell and Tregs

In [ ]:
flierprops = dict(marker='o', markersize=0, linestyle='none')


values_per_Sample = pd.crosstab(
    [sub_interest1['Baseline_comp'], sub_interest1['sample_region']],
    sub_interest1['combined_names'],
)

numerator = 'CD8_T_cell'
denominator = 'T_reg'


ratios = (
    values_per_Sample[numerator] /
    values_per_Sample[denominator].replace(0, np.nan)
).rename('ratio')

ratios_df = ratios.reset_index()

print(ratios_df.groupby('Baseline_comp')['ratio'].mean())


fig, ax = plt.subplots(figsize=(6, 5))

sns.stripplot(
    data=ratios_df,
    x='Baseline_comp',
    y='ratio',
    color='black',
    alpha=1,
    jitter=True,
    palette='Set2',
    size=10
)

ax.set_title(f'{numerator} / {denominator} ratio per Condition')
ax.set_ylabel('Log Ratio')
ax.set_xlabel('')


test_df = ratios_df.dropna(subset=['ratio'])
groups = test_df['Baseline_comp'].unique()

y_max = test_df['ratio'].max()
h = y_max * 0.05
start = y_max + h
step = h * 1.5

for i, (g1, g2) in enumerate(combinations(groups, 2)):
    a = test_df.loc[test_df['Baseline_comp'] == g1, 'ratio']
    b = test_df.loc[test_df['Baseline_comp'] == g2, 'ratio']

    if len(a) > 0 and len(b) > 0:
        stat, p = ttest_ind(a, b, equal_var=False)

        x1, x2 = list(groups).index(g1), list(groups).index(g2)
        y = start + step * i

        ax.plot(
            [x1, x1, x2, x2],
            [y, y + h, y + h, y],
            lw=1.5,
            c='k'
        )

        ax.text(
            (x1 + x2) * 0.5,
            y + h,
            f"p={p:.3g}",
            ha='center',
            va='bottom',
            color='k'
        )

plt.tight_layout()
plt.show()

In [ ]:
ratios_df

## 7: comparison based on Non-reponders with WT and mutations of interest


In [190]:
sample_to_subset = ['NEST2-10_tumor_inner','NEST2-04_tumor_inner','NEST1-08_tumor_inner','NEST1-04_tumor']
sub_interest1 = sub_df_immune[sub_df_immune['sample_region'].isin(sample_to_subset)]

In [191]:
sub_interest1['mutation_state']='WT'
sub_interest1.loc[sub_interest1['sample_region'] == 'NEST2-04_tumor_inner', 'mutation_state'] = 'BRAF_V600R'
sub_interest1.loc[sub_interest1['sample_region'] == 'NEST2-10_tumor_inner', 'mutation_state'] = 'KRAS_G12D'

/tmp/ipykernel_3977106/224637411.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sub_interest1['mutation_state']='WT'


In [ ]:
ax, results_df = t_test_plot(sub_interest1, "mutation_state")

In [193]:
results_df[results_df['p_val']<0.05]

,cell_type,comparison,BRAF_V600R_mean,KRAS_G12D_mean,t_stat,p_val,direction,WT_mean,p_adj


### Ratio of CD8_T_cell and Tregs

In [ ]:
values_per_Sample = pd.crosstab(
        [sub_interest1['mutation_state'], sub_interest1['sample_region']], 
        sub_interest1['combined_names'], 
    )

numerator = 'CD8_T_cell'
denominator = 'T_reg'

ratios = (
    values_per_Sample[numerator] /
    values_per_Sample[denominator].replace(0, np.nan)
).rename('ratio')

ratios_df = ratios.reset_index() 
print(ratios_df.groupby('mutation_state')['ratio'].mean())

# Plot
plt.figure(figsize=(6, 5))
sns.stripplot(
    data=ratios_df,
    x='mutation_state',
    y='ratio',
    color='black',
    alpha=1,
    jitter=True,
    palette='Set2',
    size=10
)
 
plt.title(f'{numerator} / {denominator} ratio per Condition')
plt.ylabel('Log Ratio')
plt.xlabel('')
plt.tight_layout()
plt.show()

In [ ]:
ratios_df

## 8: All samples


In [196]:
sub_df_immune['All'] = sub_df_immune['Internal'].values
sub_df_immune.loc[sub_df_immune['Sample_name'] == 'NEST2-06', 'All'] = 'Responder'
#sub_df_immune.loc[sub_df_immune['Sample_name'] == 'NEST1-06', 'All'] = 'Responder'
#sub_df_immune.loc[sub_df_immune['Sample_name'] == 'NEST1-05', 'All'] = 'Responder'
sub_df_immune.loc[sub_df_immune['Sample_name'] == 'NEST1-01', 'All'] = 'Responder'
sub_df_immune.loc[sub_df_immune['Sample_name'] == 'NEST2-03', 'All'] = 'Responder'
sub_df_immune.loc[sub_df_immune['Sample_name'] == 'NEST1-11', 'All'] = 'Responder'
sub_df_immune.loc[sub_df_immune['Sample_name'] == 'NEST2-01', 'All'] = 'Responder'


sub_df_immune.loc[sub_df_immune['Sample_name'] == 'NEST1-04', 'All'] = 'Non-Responder'
sub_df_immune.loc[sub_df_immune['Sample_name'] == 'NEST1-08', 'All'] = 'Non-Responder'
sub_df_immune.loc[sub_df_immune['Sample_name'] == 'NEST2-04', 'All'] = 'Non-Responder'
sub_df_immune.loc[sub_df_immune['Sample_name'] == 'NEST2-10', 'All'] = 'Non-Responder'



In [197]:
sub_df_immune['All'].unique()

array(['Responder', 'Baseline', 'Non-Responder'], dtype=object)

In [ ]:
ax, results_df = t_test_plot(sub_df_immune, "All", sample_order=["Non-Responder", "Responder","Baseline"])

### Significant res

In [199]:
results_df[results_df['p_val']<0.05]

,cell_type,comparison,Non-Responder_mean,Responder_mean,t_stat,p_val,direction,Baseline_mean,p_adj
8,CD4_Treg,Responder_vs_Baseline,NaN,0.000113,-4.452433,0.011224,up_in_Baseline,0.047451,0.302593
17,M1_Macrophages,Responder_vs_Baseline,NaN,0.019745,-2.317712,0.049146,up_in_Baseline,0.065050,0.302593


### All res

In [200]:
results_df

,cell_type,comparison,Non-Responder_mean,Responder_mean,t_stat,p_val,direction,Baseline_mean,p_adj
0,B_cell,Non-Responder_vs_Responder,0.058069,0.383097,-2.690793,0.050708,up_in_Responder,NaN,0.302593
1,B_cell,Non-Responder_vs_Baseline,0.058069,NaN,-2.166027,0.073827,up_in_Baseline,0.162742,0.302593
2,B_cell,Responder_vs_Baseline,NaN,0.383097,1.747851,0.140713,up_in_Responder,0.162742,0.302593
3,CD4_T_cell,Non-Responder_vs_Responder,0.217302,0.022718,2.151579,0.119803,up_in_Non-Responder,NaN,0.302593
4,CD4_T_cell,Non-Responder_vs_Baseline,0.217302,NaN,0.387403,0.712312,up_in_Non-Responder,0.174118,0.769297
5,CD4_T_cell,Responder_vs_Baseline,NaN,0.022718,-2.305414,0.081444,up_in_Baseline,0.174118,0.302593
6,CD4_Treg,Non-Responder_vs_Responder,0.033589,0.000113,1.907638,0.152477,up_in_Non-Responder,NaN,0.302593
7,CD4_Treg,Non-Responder_vs_Baseline,0.033589,NaN,-0.675642,0.528731,up_in_Baseline,0.047451,0.648897
8,CD4_Treg,Responder_vs_Baseline,NaN,0.000113,-4.452433,0.011224,up_in_Baseline,0.047451,0.302593
9,CD8_T_cell,Non-Responder_vs_Responder,0.147983,0.273804,-0.947256,0.385752,up_in_Responder,NaN,0.495967


In [ ]:
flierprops = dict(marker='o', markersize=0, linestyle='none')


values_per_Sample = pd.crosstab(
    [sub_df_immune['All'], sub_df_immune['sample_region']],
    sub_df_immune['combined_names'],
)

numerator = 'CD8_T_cell'
denominator = 'T_reg'


ratios = (
    values_per_Sample[numerator] /
    values_per_Sample[denominator].replace(0, np.nan)
).rename('ratio')

ratios_df = ratios.reset_index()

print(ratios_df.groupby('All')['ratio'].mean())


fig, ax = plt.subplots(figsize=(6, 5))

sns.boxplot(
    data=ratios_df,
    x='All',
    y='ratio',
    palette='Set2',
    flierprops=flierprops,
    log_scale=True,
    ax=ax
)

sns.stripplot(
    data=ratios_df,
    x='All',
    y='ratio',
    color='black',
    alpha=0.6,
    jitter=True,
    size=10,
    ax=ax
)

ax.set_title(f'{numerator} / {denominator} ratio per Condition')
ax.set_ylabel('Log Ratio')
ax.set_xlabel('')


test_df = ratios_df.dropna(subset=['ratio'])
groups = test_df['All'].unique()

y_max = test_df['ratio'].max()
h = y_max * 0.05
start = y_max + h
step = h * 1.5

for i, (g1, g2) in enumerate(combinations(groups, 2)):
    a = test_df.loc[test_df['All'] == g1, 'ratio']
    b = test_df.loc[test_df['All'] == g2, 'ratio']

    if len(a) > 0 and len(b) > 0:
        stat, p = ttest_ind(a, b, equal_var=False)

        x1, x2 = list(groups).index(g1), list(groups).index(g2)
        y = start + step * i

        ax.plot(
            [x1, x1, x2, x2],
            [y, y + h, y + h, y],
            lw=1.5,
            c='k'
        )

        ax.text(
            (x1 + x2) * 0.5,
            y + h,
            f"p={p:.3g}",
            ha='center',
            va='bottom',
            color='k'
        )

plt.tight_layout()
plt.show()